## 1. Load parametric description of the line

In [8]:
import numpy
import sys
import meshio
import vtk
from scipy.interpolate import BSpline
from scipy.spatial import cKDTree
from scipy.optimize import minimize
from collections import defaultdict


sys.path.insert(0, "./src")  # add Folder_2 path to search list


h = 1.0
id = 5
extension_factor = 0.08


pixel_spacing = numpy.loadtxt("/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/VTI_0.5_fat_AIMax/pixel_spacing.txt")[0]
offset = numpy.loadtxt("/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/offset.txt")

base_centerline = "/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/centerlines/"
base_mesh = "/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/inp files example/"


data = numpy.load(base_centerline + "centerline_id_"+str(id)+"_parametric.npz")

sx = BSpline(data["tx"], data["cx"], data["kx"])
sy = BSpline(data["ty"], data["cy"], data["ky"])
sz = BSpline(data["tz"], data["cz"], data["kz"])

mesh = meshio.read(base_mesh + "snapped_id_"+str(id)+".inp")
points = (mesh.points + offset)/pixel_spacing

cells = mesh.cells_dict["tetra"]


# --- Compute tetrahedron centers ---
centers = points[cells].mean(axis=1)  # shape: (n_cells, 3)


In [9]:
# --- Function to evaluate centerline and tangent ---
def centerline_point(u):
    return numpy.array([sx(u), sy(u), sz(u)])


def centerline_tangent(u):
    return numpy.array([sx.derivative()(u), sy.derivative()(u), sz.derivative()(u)])

In [10]:
# --- Project each center onto the centerline ---

def closest_u(center):
    """Find parameter u on spline closest to center."""
    # initial guess: middle of the spline
    u0 = 0.5*(sx.t[0] + sx.t[-1])
    
    def objective(u):
        pt = centerline_point(u)
        return numpy.sum((pt - center)**2)
    
    res = minimize(objective, u0, bounds=[(sx.t[0], sx.t[-1])])
    return res.x[0]

In [11]:
# Build connectivity dictionary for tetrahedra faces
def extract_surface_triangles(cells):
    face_dict = defaultdict(int)
    for tet in cells:
        faces = [
            tuple(sorted([tet[0], tet[1], tet[2]])),
            tuple(sorted([tet[0], tet[1], tet[3]])),
            tuple(sorted([tet[0], tet[2], tet[3]])),
            tuple(sorted([tet[1], tet[2], tet[3]])),
        ]
        for f in faces:
            face_dict[f] += 1
    # Surface triangles appear only once
    surface_faces = [list(f) for f, count in face_dict.items() if count == 1]
    return numpy.array(surface_faces)

surface_faces = extract_surface_triangles(cells)
surface_points = points[surface_faces]  # shape: (n_faces, 3, 3)

In [12]:
def triangle_normals(tri_pts):
    # tri_pts shape: (n_faces, 3, 3)
    v0 = tri_pts[:,0,:]
    v1 = tri_pts[:,1,:]
    v2 = tri_pts[:,2,:]
    n = numpy.cross(v1 - v0, v2 - v0)
    norms = numpy.linalg.norm(n, axis=1, keepdims=True)
    n = n / numpy.maximum(norms, 1e-12)
    return n

surface_normals = triangle_normals(surface_points)
surface_centers = surface_points.mean(axis=1)

# Build KDTree of surface triangle centers
tree = cKDTree(surface_centers)

N_list = []
for c in centers:
    dist, idx = tree.query(c)
    N = surface_normals[idx]
    N_list.append(N)

N_array = numpy.array(N_list)

In [13]:
# --- Compute T and N vectors for each cell ---
T_list = []

# Choose a global reference vector for normal computation
ref_vec = numpy.array([0,0,1])

for c in centers:
    u = closest_u(c)
    cl_point = centerline_point(u)
    
    # Tangent along centerline
    T = centerline_tangent(u)
    T = T / numpy.linalg.norm(T)
    
    T_list.append(T)
    N_list.append(N)

T_array = numpy.array(T_list)


print("T vectors:", T_array.shape)
print("N vectors:", N_array.shape)

T vectors: (5892, 3)
N vectors: (5892, 3)


In [14]:




# --- Prepare cell data dictionary ---
cell_data = {
    "Tangent": [T_array],  # meshio expects a list, one entry per cell block
    "Normal": [N_array]
}


# --- Create a new mesh object with cell data ---
vtk_mesh = meshio.Mesh(
    points=mesh.points,
    cells=[("tetra", cells)],
    cell_data=cell_data
)

meshio.write(
    base_mesh + "id_"+str(id)+"__h_"+str(h)+"__volume-with_direction.vtu",
    vtk_mesh
)



n_cells = len(cells)

# Create element IDs
element_ids = numpy.arange(1,n_cells+1)

# Stack everything into one array
data = numpy.column_stack([
    element_ids,
    T_array,
    N_array
])

# Header for CSV
header = "element_id,Tx,Ty,Tz,Nx,Ny,Nz"

# Save CSV
csv_filename = base_mesh + f"id_{id}_vectors_tangent_normal.csv"
numpy.savetxt(csv_filename, data,     fmt=["%d","%.6f","%.6f","%.6f","%.6f","%.6f","%.6f"], delimiter=",", header=header, comments='')

print(f"Saved CSV: {csv_filename}")

Saved CSV: /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/inp files example/id_5_vectors_tangent_normal.csv
